In [1]:
import os
import json
import time
import urllib.parse
from urllib.request import Request, urlopen
import pandas as pd
import numpy as np

# Klasör yapılarını hazırlayalım
os.makedirs('../data/cache/pubchem', exist_ok=True)
os.makedirs('../docs', exist_ok=True)

# 1. 100 ilacı belirleyelim
# ddinter_ilac.csv varsa oradan, henüz yoksa tdc_drugbank_ddi.csv üzerinden 100 benzersiz ilaç seçelim
ddinter_path = '../data/raw/ddinter_ilac.csv'
if os.path.exists(ddinter_path):
    df_ilac = pd.read_csv(ddinter_path)
    ilac_adlari = df_ilac.iloc[:, 1].dropna().unique()
else:
    df_raw = pd.read_csv('../data/raw/tdc_drugbank_ddi.csv')
    ilac_adlari = pd.concat([df_raw['Drug1_ID'], df_raw['Drug2_ID']]).dropna().unique()

np.random.seed(42)
secilen_100 = np.random.choice(ilac_adlari, size=min(100, len(ilac_adlari)), replace=False)
print(f"{len(secilen_100)} adet ilaç için PubChem SMILES pilot testi başlatılıyor...")

# 2. PubChem API sorgulama (Cache ve Rate Limiter ile)
sonuclar = []
baslangic_zamani = time.time()

for idx, ilac in enumerate(secilen_100, 1):
    ilac_temiz = str(ilac).strip()
    cache_dosya = f"../data/cache/pubchem/{ilac_temiz}.json"
    
    data = None
    if os.path.exists(cache_dosya):
        with open(cache_dosya, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{urllib.parse.quote(ilac_temiz)}/property/ConnectivitySMILES,CanonicalSMILES/JSON"
        req = Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        
        try:
            with urlopen(req, timeout=10) as resp:
                data = json.loads(resp.read().decode('utf-8'))
                with open(cache_dosya, 'w', encoding='utf-8') as f:
                    json.dump(data, f)
        except Exception:
            data = {"error": "Bulunamadı veya Zaman Aşımı"}
        
        # Saniyede en fazla 5 istek kuralı (Rate Limiter: 0.25 sn)
        time.sleep(0.25)
    
    cid_sayisi = 0
    smiles = None
    durum = "Bulunamadı"
    
    if data and "PropertyTable" in data and "Properties" in data["PropertyTable"]:
        props = data["PropertyTable"]["Properties"]
        cid_sayisi = len(props)
        if cid_sayisi > 0:
            ilk = props[0]
            smiles = ilk.get("ConnectivitySMILES") or ilk.get("CanonicalSMILES")
            durum = "Bulundu"
            
    sonuclar.append({
        "ilac": ilac_temiz,
        "durum": durum,
        "smiles": smiles,
        "cid_sayisi": cid_sayisi,
        "coklu_cid": 1 if cid_sayisi > 1 else 0
    })
    
    if idx % 20 == 0:
        print(f"-> {idx}/100 tamamlandı...")

toplam_sure = time.time() - baslangic_zamani
df_sonuc = pd.DataFrame(sonuclar)

# 3. İstatistikler ve Projeksiyon Raporu
bulunan_sayisi = (df_sonuc['durum'] == "Bulundu").sum()
coklu_cid_sayisi = (df_sonuc['coklu_cid'] == 1).sum()
ortalama_istek_suresi = toplam_sure / len(secilen_100)
toplam_1939_tahmini_dakika = (ortalama_istek_suresi * 1939) / 60

rapor_metni = f"""# İP10: PubChem SMILES Pilot Test Raporu

- **Test Edilen İlaç Sayısı:** {len(secilen_100)}
- **SMILES Bulunan İlaç Sayısı:** {bulunan_sayisi} (%{bulunan_sayisi/len(secilen_100)*100:.1f})
- **SMILES Bulunamayan / Biyolojik:** {len(secilen_100) - bulunan_sayisi}
- **Birden Fazla CID Dönen (coklu_cid=1):** {coklu_cid_sayisi}
- **100 İstek İçin Geçen Süre:** {toplam_sure:.2f} saniye (Ortalama {ortalama_istek_suresi:.2f} sn/istek)
- **1.939 İlacın Tamamı İçin Tahmini Süre:** ~{toplam_1939_tahmini_dakika:.1f} dakika

**Klinik ve Mimari Not:** 
PubChem rate limiter kuralı (5 istek/sn) ve önbellekleme (cache) mimarisi test edilmiştir. 
Tam kümenin çekilmesi yaklaşık {toplam_1939_tahmini_dakika:.1f} dakika süreceğinden, 2. hafta başında tek seferlik bir arka plan batch işlemi olarak çalıştırılması önerilmektedir.
"""

with open('../docs/smiles_pilot_raporu.md', 'w', encoding='utf-8') as f:
    f.write(rapor_metni)

print("\n--- TEST TAMAMLANDI ---")
print(f"Bulunan SMILES: {bulunan_sayisi}/{len(secilen_100)}")
print(f"Toplam süre: {toplam_sure:.1f} sn")
print(f"1.939 ilaç için tahmini süre: ~{toplam_1939_tahmini_dakika:.1f} dakika")
print("Rapor docs/smiles_pilot_raporu.md dosyasına kaydedildi.")


100 adet ilaç için PubChem SMILES pilot testi başlatılıyor...
-> 20/100 tamamlandı...
-> 40/100 tamamlandı...
-> 60/100 tamamlandı...
-> 80/100 tamamlandı...
-> 100/100 tamamlandı...

--- TEST TAMAMLANDI ---
Bulunan SMILES: 100/100
Toplam süre: 54.7 sn
1.939 ilaç için tahmini süre: ~17.7 dakika
Rapor docs/smiles_pilot_raporu.md dosyasına kaydedildi.
